# 12 — DenseNet121 Extra Seed 3407 + Multi-Seed Ensemble

Mục tiêu: kiểm tra độ ổn định theo seed của recipe finalist hiện tại mà không mở thêm một đợt tuning lớn.

Chạy thêm đúng **1 seed mới**:
- recipe: DenseNet121, grayscale → RGB, 224px
- batch 8 + grad accumulation 2
- base seed = 3407
- 5-fold OOF

Sau đó so:
1. historical DenseNet OOF
2. grayscale seed 2026 OOF
3. grayscale seed 3407 OOF
4. ensemble 3 nguồn OOF

Nếu seed 3407 / ensemble không tạo gain đáng kể, dừng multi-seed ở đây. Nếu gain rõ, lúc đó mới cân nhắc seed thứ ba.

Internal holdout vẫn niêm phong.


In [ ]:
print('>>> CELL 1 START: config', flush=True)
import os
os.environ['TORCH_HOME']=r'D:\AI_Cache\torch'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

from pathlib import Path
import json,time,random,gc
import numpy as np,pandas as pd
from PIL import Image,ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models
from sklearn.metrics import confusion_matrix,f1_score,roc_auc_score

SEED=3407
ROOT=Path(r'D:\medicine')
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
OUT=ROOT/'cv_outputs'/'densenet121_224_gray_b8a2_seed3407'
OUT.mkdir(parents=True,exist_ok=True)
OLD_OOF=ROOT/'cv_outputs'/'densenet121_224'/'oof_predictions.csv'
GRAY2026_OOF=ROOT/'cv_outputs'/'densenet121_224_gray_b8a2'/'oof_predictions.csv'

IMAGE_SIZE=224
BATCH_SIZE=8
ACCUM=2
EPOCHS=8
PATIENCE=3
LR=1e-4
WD=1e-4
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
torch.backends.cudnn.benchmark=USE_AMP

print('Device:',DEVICE,flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Base seed:',SEED,flush=True)
print('Batch:',BATCH_SIZE,'| accum:',ACCUM,'| effective:',BATCH_SIZE*ACCUM,flush=True)
print('Historical OOF exists:',OLD_OOF.exists(),flush=True)
print('Gray-2026 OOF exists:',GRAY2026_OOF.exists(),flush=True)


In [ ]:
print('>>> CELL 2 START: locked dev + grayscale cache', flush=True)
manifest=pd.read_csv(MANIFEST)
dev=manifest[manifest.split=='dev'].copy().reset_index(drop=True)
holdout=manifest[manifest.split=='holdout'].copy().reset_index(drop=True)
assert len(dev)==4362 and len(holdout)==770
assert set(dev.sha256).isdisjoint(set(holdout.sha256))
dev['cache_idx']=np.arange(len(dev))

class PadToSquare:
    def __call__(self,img):
        w,h=img.size;s=max(w,h)
        l=(s-w)//2;r=s-w-l;t=(s-h)//2;b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

pad=PadToSquare()
buf=[];t0=time.time()
for i,row in dev.iterrows():
    with Image.open(row.path) as im:
        im=im.convert('L').convert('RGB')
        im=pad(im).resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BILINEAR)
        buf.append(np.asarray(im,dtype=np.uint8))
    if (i+1)%500==0 or i+1==len(dev):
        print(f'  cached {i+1}/{len(dev)}',flush=True)
DEV_CACHE=np.stack(buf)
del buf
print(f'Cache {DEV_CACHE.shape} | {DEV_CACHE.nbytes/1024**3:.2f} GiB | {time.time()-t0:.0f}s',flush=True)
print('SEALED holdout:',len(holdout),flush=True)


In [ ]:
print('>>> CELL 3 START: helpers', flush=True)
MEAN=[0.485,0.456,0.406];STD=[0.229,0.224,0.225]
L2I={'Normal':0,'Pneumonia':1}

train_tf=transforms.Compose([
    transforms.RandomAffine(degrees=5,translate=(0.02,0.02),scale=(0.95,1.05),fill=0),
    transforms.ColorJitter(brightness=0.08,contrast=0.08),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
val_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

class CachedDS(Dataset):
    def __init__(self,df,tf): self.df=df.reset_index(drop=True);self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        return self.tf(Image.fromarray(DEV_CACHE[int(r.cache_idx)])),L2I[r.label],r.filename

def score(y,p,t=.5,auc=None):
    y=np.asarray(y,int);p=np.asarray(p,float);pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y);se=tp/max(tp+fn,1);sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    if auc is None: auc=roc_auc_score(y,p)
    return {'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),'Specificity':float(sp),
            'F1':float(f1),'AUROC':float(auc),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
            'threshold':float(t)}

@torch.no_grad()
def predict(model,loader):
    model.eval();ys=[];ps=[];names=[]
    for x,y,n in loader:
        x=x.to(DEVICE)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            z=model(x)
        ps.extend(torch.softmax(z.float(),1)[:,1].cpu().numpy())
        ys.extend(y.numpy());names.extend(list(n))
    return np.asarray(ys),np.asarray(ps),names

def make_model():
    m=models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    m.classifier=nn.Linear(m.classifier.in_features,2)
    return m.to(DEVICE)


In [ ]:
print('>>> CELL 4 START: 5-fold training', flush=True)
all_oof=[];fold_rows=[]
for fold in range(5):
    print('\n'+'='*70,flush=True);print('FOLD',fold,flush=True);print('='*70,flush=True)
    seed=SEED+fold
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if USE_AMP: torch.cuda.manual_seed_all(seed)

    tr=dev[dev.cv_fold!=fold].copy().reset_index(drop=True)
    va=dev[dev.cv_fold==fold].copy().reset_index(drop=True)
    assert set(tr.sha256).isdisjoint(set(va.sha256))

    tl=DataLoader(CachedDS(tr,train_tf),batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=False)
    vl=DataLoader(CachedDS(va,val_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=False)

    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

    model=make_model()
    c=tr.label.value_counts();n0=int(c.Normal);n1=int(c.Pneumonia);tot=n0+n1
    cw=torch.tensor([tot/(2*n0),tot/(2*n1)],dtype=torch.float32,device=DEVICE)
    crit=nn.CrossEntropyLoss(weight=cw)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)

    best=-1;best_epoch=0;best_state=None;stale=0;hist=[]
    for epoch in range(1,EPOCHS+1):
        t0=time.time();model.train();opt.zero_grad(set_to_none=True);ls=0.;seen=0
        for bi,(x,y,_) in enumerate(tl,1):
            x=x.to(DEVICE);y=y.to(DEVICE)
            with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
                z=model(x);raw=crit(z,y);loss=raw/ACCUM
            scaler.scale(loss).backward()
            if bi%ACCUM==0 or bi==len(tl):
                scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
            ls+=float(raw.item())*len(x);seen+=len(x)
            if bi%100==0 or bi==len(tl):
                print(f'  fold {fold} epoch {epoch:02d} batch {bi}/{len(tl)} | loss={ls/seen:.4f}',flush=True)
        sched.step()
        yv,pv,_=predict(model,vl);m=score(yv,pv,.5);sec=time.time()-t0
        hist.append({'fold':fold,'epoch':epoch,'train_loss':ls/seen,'seconds':sec,**m})
        print(f"Fold {fold} Epoch {epoch:02d} | P={m['P']:.5f} | A={m['Accuracy']:.5f} | Se={m['Sensitivity']:.5f} | Sp={m['Specificity']:.5f} | F1={m['F1']:.5f} | AUROC={m['AUROC']:.6f} | {sec:.0f}s",flush=True)
        if m['P']>best+1e-6:
            best=m['P'];best_epoch=epoch;stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if stale>=PATIENCE:
                print('Early stopping',flush=True);break

    model.load_state_dict(best_state)
    yv,pv,names=predict(model,vl);fm=score(yv,pv,.5)
    pred=pd.DataFrame({'filename':names,'y_true':yv,'p_pneumonia':pv,'fold':fold})
    pred.to_csv(OUT/f'fold{fold}_predictions.csv',index=False)
    pd.DataFrame(hist).to_csv(OUT/f'fold{fold}_history.csv',index=False)
    torch.save({'architecture':'densenet121','preprocessing':'force_grayscale_then_rgb',
                'batch_size':BATCH_SIZE,'grad_accum':ACCUM,'fold':fold,'best_epoch':best_epoch,
                'state_dict':best_state},OUT/f'fold{fold}_best.pt')
    fold_rows.append({'fold':fold,'best_epoch':best_epoch,**fm});all_oof.append(pred)

    del model,opt,sched,scaler,best_state,tl,vl
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

print('>>> ALL FOLDS FINISHED',flush=True)


In [ ]:
print('>>> CELL 5 START: OOF + multi-seed ensemble', flush=True)
oof=pd.concat(all_oof,ignore_index=True)
assert len(oof)==4362 and oof.filename.nunique()==4362
oof.to_csv(OUT/'oof_predictions.csv',index=False)

fold_table=pd.DataFrame(fold_rows)
fold_table.to_csv(OUT/'fold_summary.csv',index=False)
display(fold_table)

fold_mean=float(fold_table.P.mean())
fold_std=float(fold_table.P.std(ddof=1))
base=score(oof.y_true,oof.p_pneumonia,.5)
auc=float(roc_auc_score(oof.y_true,oof.p_pneumonia))
ths=np.round(np.arange(.05,.951,.001),3)
scan=pd.DataFrame([score(oof.y_true,oof.p_pneumonia,float(t),auc) for t in ths])
br=scan.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
bt=float(br.threshold)
tuned=score(oof.y_true,oof.p_pneumonia,bt,auc)
scan.to_csv(OUT/'oof_threshold_scan.csv',index=False)

print('Seed 3407 OOF @0.5:',base,flush=True)
print('Seed 3407 OOF tuned:',tuned,flush=True)

multi_summary=None

if OLD_OOF.exists() and GRAY2026_OOF.exists():
    old=pd.read_csv(OLD_OOF)[['filename','y_true','p_pneumonia']].rename(columns={'p_pneumonia':'p_old'})
    g26=pd.read_csv(GRAY2026_OOF)[['filename','y_true','p_pneumonia','fold']].rename(columns={'p_pneumonia':'p_g26'})
    g34=oof[['filename','y_true','p_pneumonia','fold']].rename(columns={'p_pneumonia':'p_g34','fold':'fold_new'})

    merged=g26.merge(old,on=['filename','y_true'],validate='one_to_one')
    merged=merged.merge(g34,on=['filename','y_true'],validate='one_to_one')
    assert len(merged)==4362
    assert (merged['fold']==merged['fold_new']).all()

    def best_threshold_for_probs(p):
        a=float(roc_auc_score(merged.y_true,p))
        rows=[score(merged.y_true,p,float(t),a) for t in ths]
        df=pd.DataFrame(rows)
        b=df.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
        t=float(b.threshold)
        return t,score(merged.y_true,p,t,a)

    # Individual references
    old_t,old_m=best_threshold_for_probs(merged.p_old.values)
    g26_t,g26_m=best_threshold_for_probs(merged.p_g26.values)
    g34_t,g34_m=best_threshold_for_probs(merged.p_g34.values)

    # Simple 2-seed grayscale average
    p_seedavg=.5*merged.p_g26.values+.5*merged.p_g34.values
    seedavg_t,seedavg_m=best_threshold_for_probs(p_seedavg)

    # Coarse 3-way weight search. Weights sum to 1.
    coarse=[]
    coarse_th=np.round(np.arange(.10,.901,.005),3)
    for w_old in np.round(np.arange(0,1.001,.05),3):
        for w26 in np.round(np.arange(0,1.001-w_old,.05),3):
            w34=round(1.0-w_old-w26,3)
            if w34 < -1e-9:
                continue
            p=w_old*merged.p_old.values+w26*merged.p_g26.values+w34*merged.p_g34.values
            a=float(roc_auc_score(merged.y_true,p))
            for t in coarse_th:
                m=score(merged.y_true,p,float(t),a)
                coarse.append({'w_old':float(w_old),'w_g26':float(w26),'w_g34':float(w34),**m})

    cdf=pd.DataFrame(coarse)
    cdf.to_csv(OUT/'multiseed_ensemble_coarse.csv',index=False)
    cb=cdf.sort_values(['P','w_old','w_g26','threshold'],ascending=[False,True,True,True]).iloc[0]

    # Fine search around coarse best.
    wo0=float(cb.w_old); w260=float(cb.w_g26); t0=float(cb.threshold)
    fine=[]
    wo_vals=np.round(np.arange(max(0,wo0-.06),min(1,wo0+.06)+1e-9,.01),3)
    w26_vals=np.round(np.arange(max(0,w260-.06),min(1,w260+.06)+1e-9,.01),3)
    fine_th=np.round(np.arange(max(.01,t0-.03),min(.99,t0+.03)+1e-9,.001),3)

    for w_old in wo_vals:
        for w26 in w26_vals:
            w34=1.0-w_old-w26
            if w34 < 0 or w34 > 1:
                continue
            p=w_old*merged.p_old.values+w26*merged.p_g26.values+w34*merged.p_g34.values
            a=float(roc_auc_score(merged.y_true,p))
            for t in fine_th:
                m=score(merged.y_true,p,float(t),a)
                fine.append({'w_old':float(w_old),'w_g26':float(w26),'w_g34':float(w34),**m})

    fdf=pd.DataFrame(fine)
    fdf.to_csv(OUT/'multiseed_ensemble_fine.csv',index=False)
    fb=fdf.sort_values(['P','w_old','w_g26','threshold'],ascending=[False,True,True,True]).iloc[0]

    wo=float(fb.w_old); w26=float(fb.w_g26); w34=float(fb.w_g34); et=float(fb.threshold)
    ep=wo*merged.p_old.values+w26*merged.p_g26.values+w34*merged.p_g34.values
    em=score(merged.y_true,ep,et,float(roc_auc_score(merged.y_true,ep)))

    fold_ps=[]
    for f in range(5):
        mask=merged['fold'].values==f
        fold_ps.append(score(merged.y_true.values[mask],ep[mask],et)['P'])

    multi_summary={
        'historical_dense':{'threshold':old_t,'metrics':old_m},
        'gray_seed2026':{'threshold':g26_t,'metrics':g26_m},
        'gray_seed3407':{'threshold':g34_t,'metrics':g34_m},
        'gray_two_seed_equal_average':{'threshold':seedavg_t,'metrics':seedavg_m},
        'best_three_way_ensemble':{
            'old_weight':wo,
            'gray2026_weight':w26,
            'gray3407_weight':w34,
            'threshold':et,
            'metrics':em,
            'fold_P_mean':float(np.mean(fold_ps)),
            'fold_P_std':float(np.std(fold_ps,ddof=1)),
            'fold_P_values':[float(x) for x in fold_ps]
        }
    }

summary={
    'model':'DenseNet121',
    'recipe':'grayscale_224_batch8_accum2',
    'base_seed':SEED,
    'fold_P_mean':fold_mean,
    'fold_P_std':fold_std,
    'oof_at_0_5':base,
    'oof_best_threshold':bt,
    'oof_tuned':tuned,
    'multi_seed_comparison':multi_summary,
    'holdout_used':False
}

with open(OUT/'oof_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2)

print('\nFINAL SUMMARY',flush=True)
print(json.dumps(summary,indent=2),flush=True)
